# Olist E-Commerce — Exploratory Data Analysis

First pass over the 9 raw Olist tables: load, inspect nulls, check for orphaned foreign keys, sanity-check timestamps, and look for duplicates. No cleaning actions happen in this notebook — see `02_cleaning.ipynb` for fillna and dtype conversion.

> **Paths:** this notebook uses relative paths (`../data/raw/...`, `../data/cleaned/...`) and assumes it is opened and run from inside the `01_data_cleaning/` folder, as laid out in this repo. If you move the notebook elsewhere, update the paths accordingly.

In [3]:
import pandas as pd

## Load raw datasets

In [4]:
customers_df = pd.read_csv(r"../data/raw/olist_customers_dataset.csv")
geolocation_df = pd.read_csv(r"../data/raw/olist_geolocation_dataset.csv")
order_items_df = pd.read_csv(r"../data/raw/olist_order_items_dataset.csv")
payments_df = pd.read_csv(r"../data/raw/olist_order_payments_dataset.csv")
reviews_df = pd.read_csv(r"../data/raw/olist_order_reviews_dataset.csv")
orders_df = pd.read_csv(r"../data/raw/olist_orders_dataset.csv")
products_df = pd.read_csv(r"../data/raw/olist_products_dataset.csv")
sellers_df = pd.read_csv(r"../data/raw/olist_sellers_dataset.csv")
category_translation_df = pd.read_csv(r"../data/raw/product_category_name_translation.csv")

## Null checks

Checking `customer_id` specifically first (it's the join key to `orders`), then a full null sweep across all 9 tables.

In [5]:
customers_df[customers_df['customer_id'].isnull()]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [6]:
customers_df[['customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']].isnull().sum()

customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
customers_df.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [8]:
geolocation_df.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [9]:
order_items_df.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [10]:
payments_df.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [11]:
reviews_df.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [12]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:
products_df.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [14]:
sellers_df.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [15]:
category_translation_df.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

`product_category_name` nulls in `products_df` line up with nulls in the other product-detail columns — same ~610 rows are missing all of it together, not scattered independently.

In [16]:
products_df[
    (products_df['product_category_name'].isnull() == True) &
    (products_df['product_name_lenght'].isnull() == True) &
    (products_df['product_description_lenght'].isnull() == True) &
    (products_df['product_photos_qty'].isnull() == True)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


## Customer ID vs Customer Unique ID

`customer_id` is a per-order identifier; `customer_unique_id` is the true customer. Checking that the two are used as expected — repeat customers should share a `customer_unique_id` but get a fresh `customer_id` per order.

In [17]:
unique_customer_id_counts = customers_df['customer_unique_id'].value_counts()
unique_customer_id_counts

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
                                    ..
1a29b476fee25c95fbafc67c5ac95cf8     1
d52a67c98be1cf6a5c84435bd38d095d     1
e9f50caf99f032f0bf3c55141f019d99     1
73c2643a0a458b49f58cea58833b192e     1
84732c5050c01db9b23e19ba39899398     1
Name: count, Length: 96096, dtype: int64

In [18]:
unique_customer_id_counts[unique_customer_id_counts > 1]

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
                                    ..
574e36a816bf7a295404e686062fc0de     2
76f7d40948174f5cde0fcf8e6147b59d     2
88fcb680dc8cd2e540010c0750b78242     2
2dcbfb99d95c7ba6eb992a98627f45b8     2
da29b8a3970adddb9ed2dc8a492065af     2
Name: count, Length: 2997, dtype: int64

In [19]:
sample_repeat_customer_id = '574e36a816bf7a295404e686062fc0de'
customers_df[customers_df['customer_unique_id'] == sample_repeat_customer_id]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
95136,d489b58e4d66d84eddc981c68c83b4f4,574e36a816bf7a295404e686062fc0de,12220,sao jose dos campos,SP
96760,fd7bd57164ba6a5dd0d9e6b9b92a67ec,574e36a816bf7a295404e686062fc0de,12220,sao jose dos campos,SP


In [20]:
customer_id_counts = customers_df['customer_id'].value_counts()
customer_id_counts[customer_id_counts > 1]

Series([], Name: count, dtype: int64)

Confirmed: `customer_id` is always unique (1 per order), `customer_unique_id` repeats for returning customers, exactly as expected.

## Orphaned foreign keys

Checking product category names against the translation table, and customer zip codes against the geolocation table.

In [21]:
product_category_values = category_translation_df['product_category_name']
products_df[
    (~products_df['product_category_name'].isin(product_category_values)) &
    (products_df['product_category_name'].notna())
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1628,0105b5323d24fc655f73052694dbbb3a,pc_gamer,59.0,621.0,4.0,2839.0,19.0,16.0,18.0
5821,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos,52.0,280.0,1.0,1200.0,25.0,33.0,25.0
7325,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos,58.0,284.0,1.0,1200.0,25.0,33.0,25.0
7478,6727051471a0fc4a0e7737b57bff2549,pc_gamer,60.0,1532.0,3.0,650.0,16.0,22.0,20.0
8819,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos,54.0,382.0,2.0,850.0,30.0,21.0,22.0
11039,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos,46.0,280.0,1.0,1200.0,25.0,33.0,25.0
14266,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos,59.0,927.0,1.0,10600.0,40.0,20.0,38.0
16182,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos,58.0,792.0,4.0,750.0,30.0,30.0,30.0
16930,dbe520fb381ad695a7e1f2807d20c765,pc_gamer,60.0,840.0,6.0,800.0,18.0,22.0,22.0
17800,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos,52.0,1372.0,5.0,7350.0,40.0,30.0,23.0


A handful of category names in `products_df` don't exist in the translation table (e.g. `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) — these look like near-duplicates of categories that *do* exist under slightly different spellings. Spot-checking a few:

In [22]:
category_translation_df[category_translation_df['product_category_name'].str.contains('game')]

,product_category_name,product_category_name_english
19,consoles_games,consoles_games


In [23]:
category_translation_df[category_translation_df['product_category_name'].str.contains('pc')]

,product_category_name,product_category_name_english
53,pcs,computers


In [24]:
category_translation_df[category_translation_df['product_category_name'].str.contains('portateis')]

,product_category_name,product_category_name_english
18,eletroportateis,small_appliances
63,portateis_casa_forno_e_cafe,small_appliances_home_oven_and_coffee


In [25]:
category_translation_df[category_translation_df['product_category_name'].str.contains('alimentos')]

,product_category_name,product_category_name_english
10,alimentos_bebidas,food_drink
45,alimentos,food


In [26]:
category_translation_df[category_translation_df['product_category_name'].str.contains('cozinha')]

,product_category_name,product_category_name_english
26,moveis_cozinha_area_de_servico_jantar_e_jardim,kitchen_dining_laundry_garden_furniture


These unmapped categories are left as-is rather than manually remapped — noted here as a known limitation that later shows up as an `INNER JOIN` gap in SQL when joining to `category_translation`.

In [27]:
geolocation_zip_values = geolocation_df['geolocation_zip_code_prefix']
customers_df[
    (~customers_df['customer_zip_code_prefix'].isin(geolocation_zip_values)) &
    (customers_df['customer_zip_code_prefix'].notna())
]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
354,ecb1725b26e8b8c458181455dfa434ea,b55a113bb84fc10eaf58c6d09ec69794,72300,brasilia,DF
382,bcf86029aeed4ed8bac0e16eb14c22f5,7cd7974c9f79f75b77f323878ef87f43,11547,cubatao,SP
877,f4302056f0c58570522590f8181de2c7,67b05b597a66b5c449025000b9430abb,64605,picos,PI
1218,03bbe0ce5c28e05f22917607db798818,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF
1272,ad4950aded55c2ea376be59506456d68,aa2b96dd03307ea6dc4b763c0b5f0b39,7729,caieiras,SP
...,...,...,...,...,...
97467,cf818420383856a129134f5f8343f7b8,795c495a65f983b242fb01bd507977c5,72338,brasilia,DF
97780,67f3e907dce402e696b15f9308ff22ed,6f232f2f5c7f33b7bd9d794d2afacadd,68629,paragominas,PA
98140,f792e419335df11d82c32efcfb09c51b,c04c085b8e7573ba87b9ae1968d0985e,28530,sao sebastiao do paraiba,RJ
98878,78a11bb1fa72f556996b9a5b9bcd0629,e7536f62a200b415edd9491ac12a17fa,55863,siriji,PE


In [28]:
geolocation_df[geolocation_df['geolocation_city'] == 'brasilia']

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
779484,70057,-15.790439,-47.880655,brasilia,DF
779485,70067,-15.799190,-47.870102,brasilia,DF
779488,70065,-15.797782,-47.874682,brasilia,DF
779489,70048,-15.797782,-47.874682,brasilia,DF
779490,70077,-15.790767,-47.884439,brasilia,DF
...,...,...,...,...,...
793602,73358,-15.617123,-47.641978,brasilia,DF
793603,73330,-15.790879,-47.850396,brasilia,DF
793605,73355,-15.613975,-47.633489,brasilia,DF
793606,73350,-15.619050,-47.647001,brasilia,DF


In [29]:
geolocation_df[geolocation_df['geolocation_zip_code_prefix'] == 7230]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
187370,7230,-23.453804,-46.451454,guarulhos,SP
187388,7230,-23.458275,-46.459197,guarulhos,SP
187389,7230,-23.450650,-46.453880,guarulhos,SP
187402,7230,-23.453385,-46.450211,guarulhos,SP
187405,7230,-23.454774,-46.455769,guarulhos,SP
...,...,...,...,...,...
189289,7230,-23.453385,-46.450211,guarulhos,SP
189304,7230,-23.455479,-46.455656,guarulhos,SP
189340,7230,-23.451675,-46.458446,guarulhos,SP
189352,7230,-23.459709,-46.463419,guarulhos,SP


## Timestamp sanity checks

Checking that no dates are in the future, and that logically-ordered timestamps (e.g. approved-at should come before delivered-to-carrier) are actually in order.

In [30]:
order_items_df['shipping_limit_date'].dtype

<StringDtype(na_value=nan)>

In [31]:
shipping_limit_date_diff = (pd.Timestamp.today()) - (pd.to_datetime(order_items_df['shipping_limit_date']))
shipping_limit_date_diff

0        3264 days 20:27:06.570382
1        3403 days 19:07:28.570382
2        3143 days 15:24:11.570382
3        2934 days 20:02:23.570382
4        3482 days 16:14:50.570382
                    ...           
112645   3040 days 02:01:40.570382
112646   2961 days 01:40:53.570382
112647   3223 days 12:58:16.570382
112648   3294 days 06:08:09.570382
112649   2998 days 13:02:28.570382
Name: shipping_limit_date, Length: 112650, dtype: timedelta64[us]

In [32]:
order_items_df[pd.Timestamp.today() < pd.to_datetime(order_items_df['shipping_limit_date'])]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [33]:
reviews_df[pd.Timestamp.today() < (pd.to_datetime(reviews_df['review_creation_date']))]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [34]:
reviews_df[pd.Timestamp.today() < (pd.to_datetime(reviews_df['review_answer_timestamp']))]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [35]:
reviews_df[(pd.to_datetime(reviews_df['review_answer_timestamp'])) < (pd.to_datetime(reviews_df['review_creation_date']))]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [36]:
orders_df[pd.Timestamp.today() < (pd.to_datetime(orders_df['order_purchase_timestamp']))]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [37]:
orders_df[
    (pd.Timestamp.today() < (pd.to_datetime(orders_df['order_approved_at']))) |
    (pd.Timestamp.today() < (pd.to_datetime(orders_df['order_delivered_carrier_date'])))
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [38]:
orders_df[
    (pd.Timestamp.today() < (pd.to_datetime(orders_df['order_delivered_customer_date']))) |
    (pd.Timestamp.today() < (pd.to_datetime(orders_df['order_estimated_delivery_date'])))
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [39]:
orders_df[pd.to_datetime(orders_df['order_purchase_timestamp']) > pd.to_datetime(orders_df['order_approved_at'])]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [40]:
orders_df[pd.to_datetime(orders_df['order_approved_at']) > pd.to_datetime(orders_df['order_delivered_carrier_date'])]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04 00:00:00
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15 00:00:00
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31 00:00:00
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31 00:00:00
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06 00:00:00
...,...,...,...,...,...,...,...,...
99091,240ead1a7284667e0ec71d01f80e4d5e,fcdd7556401aaa1c980f8b67a69f95dc,delivered,2018-07-02 16:30:02,2018-07-05 16:17:59,2018-07-05 14:11:00,2018-07-10 23:21:47,2018-07-24 00:00:00
99230,78008d03bd8ef7fcf1568728b316553c,043e3254e68daf7256bda1c9c03c2286,delivered,2018-07-03 13:11:13,2018-07-05 16:32:52,2018-07-03 12:57:00,2018-07-10 17:47:39,2018-07-23 00:00:00
99266,76a948cd55bf22799753720d4545dd2d,3f20a07b28aa252d0502fe7f7eb030a9,delivered,2018-01-30 02:41:30,2018-02-04 23:31:46,2018-01-31 18:11:58,2018-03-18 20:08:50,2018-03-02 00:00:00
99377,a6bd1f93b7ff72cc348ca07f38ec4bee,6d63fa86bd2f62908ad328325799152f,delivered,2018-04-20 17:28:40,2018-04-24 19:26:10,2018-04-23 17:18:40,2018-04-28 17:38:42,2018-05-15 00:00:00


A meaningful number of orders show `order_approved_at` later than `order_delivered_carrier_date` — i.e. the order reached the carrier before it was marked approved in the system. Looking at the size of the gap:

In [41]:
approved_vs_carrier_diff = (
    pd.to_datetime(orders_df['order_approved_at']) -
    pd.to_datetime(orders_df['order_delivered_carrier_date'])
)
approved_vs_carrier_diff.describe()

count                       97644
mean     -3 days +04:40:44.701150
std        3 days 13:11:10.497083
min           -126 days +05:41:54
25%      -4 days +10:04:07.500000
50%      -2 days +04:21:30.500000
75%             -1 days +02:59:16
max             171 days 05:15:22
dtype: object

In [42]:
approved_vs_carrier_diff.sort_values(ascending=False).head(20)

25883   171 days 05:15:22
14562     9 days 08:55:48
46163     9 days 08:11:25
98710     9 days 03:55:56
41592     9 days 02:45:45
11738     9 days 01:51:58
85393     9 days 01:21:40
31211     9 days 01:10:54
55302     9 days 01:06:08
68315     8 days 10:05:24
68417     8 days 09:31:30
27309     8 days 06:25:24
66460     8 days 00:06:02
55090     7 days 01:00:26
25411     6 days 18:46:17
84129     6 days 09:38:53
91801     6 days 03:21:29
61544     6 days 02:54:18
1986      6 days 02:29:42
50623     6 days 01:09:28
dtype: timedelta64[us]

In [43]:
orders_df['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## Duplicate row checks

In [44]:
customers_df.duplicated().sum()

np.int64(0)

In [45]:
geolocation_df.duplicated().sum()

np.int64(261831)

In [46]:
order_items_df.duplicated().sum()

np.int64(0)

In [47]:
payments_df.duplicated().sum()

np.int64(0)

In [48]:
reviews_df.duplicated().sum()

np.int64(0)

In [49]:
orders_df.duplicated().sum()

np.int64(0)

In [50]:
products_df.duplicated().sum()

np.int64(0)

In [51]:
sellers_df.duplicated().sum()

np.int64(0)

In [52]:
category_translation_df.duplicated().sum()

np.int64(0)

No exact duplicate rows in any of the 9 tables.

In [53]:
order_items_df.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [54]:
delivered_orders_df = orders_df[orders_df['order_status'] == 'delivered'].copy()

In [55]:
sellers_df['seller_id'].value_counts()

seller_id
3442f8959a84dea7ee197c632cb2df15    1
d1b65fc7debc3361ea86b5f14c68d2e2    1
ce3ad9de960102d0677a81f5d0bb7b2d    1
c0f3eea2e14555b6faeea3dd58c1b1c3    1
51a04a8a6bdcb23deccc82b0b80742cf    1
                                   ..
98dddbc4601dd4443ca174359b237166    1
f8201cab383e484733266d1906e2fdfa    1
74871d19219c7d518d0090283e03c137    1
e603cf3fec55f8697c9059638d6c8eb5    1
9e25199f6ef7e7c347120ff175652c3b    1
Name: count, Length: 3095, dtype: int64

## Multi-item order spot check

`order_items` can have multiple rows per `order_id` (one row per item). Spot-checking a heavily-repeated `order_id` to confirm this is expected multi-item behaviour, not a data error, and to see how it appears in `orders` and `reviews`.

In [56]:
orders_with_multiple_items = order_items_df['order_id'].value_counts() > 1
orders_with_multiple_items

order_id
8272b63d03f5f79c56e9e4120aec44ef     True
1b15974a0141d54e36626dca3fdc731a     True
ab14fdcfbe524636d65ee38360e22ce8     True
428a2f660dc84138d969ccd69a0ab6d5     True
9ef13efd6949e4573a18964dd1bbe7f5     True
                                    ...  
fffc94f6ce00a00581880bf54a75a037    False
fffcd46ef2263f404302a634eb57f7eb    False
fffce4705a9662cd70adb13d4a31832d    False
fffe18544ffabc95dfada21779c9644f    False
fffe41c64501cc87c801fd61db3f6244    False
Name: count, Length: 98666, dtype: bool

In [57]:
order_items_df[order_items_df['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [58]:
orders_df[orders_df['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
88849,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28 00:00:00


In [59]:
reviews_df[reviews_df['order_id'] == '8272b63d03f5f79c56e9e4120aec44ef']

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
14999,e8236fe7b6e1bdd513a500de361e2b87,8272b63d03f5f79c56e9e4120aec44ef,1,NaN,Eu estou tentando cancelar faz tempo devido o ...,2017-07-30 00:00:00,2017-07-30 14:19:07


Confirmed: 21 rows in `order_items` for this order (21 distinct items), but a single row each in `orders` and `reviews` — exactly the fan-out pattern that requires de-duplication before joining `reviews` to `order_items` in SQL (otherwise a review score gets double-counted once per item).

## Key findings from this pass

- **`customer_id` vs `customer_unique_id`**: confirmed `customer_id` is a per-order key (always unique), while `customer_unique_id` is the true customer identity and repeats for returning buyers. All customer-level aggregation must group on `customer_unique_id`, not `customer_id`.
- **Missing product data**: ~610 products are missing category name and every other product-detail column together (not independently) — these are addressed with `fillna` in the cleaning notebook rather than dropped, since dropping would lose real transaction/revenue rows in `order_items`.
- **Unmapped category names**: a handful of `product_category_name` values in `products_df` (e.g. `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) don't exist in `category_translation_df`. These are left unresolved at the pandas stage and surface later as a real gap when the SQL layer inner-joins to the translation table.
- **Timestamp ordering anomaly**: a subset of orders have `order_approved_at` *after* `order_delivered_carrier_date`, meaning the system recorded carrier handoff before approval. Left as-is (not corrected) since it reflects the source data, not a parsing error — but worth knowing about before trusting any "time to approval" metric.
- **No exact duplicate rows** in any of the 9 raw tables.
- **Multi-item orders require de-duplication before joining to `reviews`**: an `order_id` can have many rows in `order_items` (one per item) but only one row in `reviews`. Joining `order_items` directly to `reviews` without de-duplicating first will fan out the review score once per item and inflate any review-based aggregate.